# Manutenção Preditiva e Prescritiva — Classificação de FalhasNotebook de documentação do pipeline: cada etapa é explicada e reproduzida com o código realusado no projeto.**O problema.** A partir de sinais de vibração e temperatura de um motor, prever qual falha estáocorrendo. É uma classificação multiclasse sobre séries temporais.**O caminho.** Os dados brutos são uma série contínua de leituras. Eles passam porordenação → segmentação → normalização de rótulos → seleção de sinais → recorte em janelas →resumo em features → modelo → validação.**Roteiro do notebook**| Seção | Conteúdo ||---|---|| 1 | Preparação do ambiente e carga dos dados || 2 | Etapa 1 — Carregamento e ordenação temporal || 3 | Etapa 2 — Segmentação || 4 | Etapa 3 — Normalização de rótulos || 5 | Etapa 4 — Seleção de sinais || 6 | Etapa 5 — Fatiar em janelas || 7 | Etapa 6 — Resumir cada janela em números || 8 | Etapa 7 — Modelo || 9 | Etapa 8 — Validação cruzada (e por que existem duas) || 10 | Experimento — varredura do tamanho de janela || 11 | Custo de descarte por tamanho de janela || 12 | Conclusões, decisão tomada e pendências |

---## 1. Preparação do ambiente e carga dos dadosO notebook depende apenas de `pandas`, `numpy` e `scikit-learn`, todos já presentes no Colab.

In [ ]:
import warningswarnings.filterwarnings("ignore")from pathlib import Pathimport numpy as npimport pandas as pdfrom sklearn.ensemble import RandomForestClassifierfrom sklearn.metrics import accuracy_score, classification_report, confusion_matrixfrom sklearn.model_selection import StratifiedGroupKFold, StratifiedKFoldpd.set_option("display.max_columns", 60)pd.set_option("display.width", 200)print("pandas", pd.__version__, "| numpy", np.__version__)

### 1.1 Obter o `banner.csv`O arquivo tem ~32 MB e não acompanha o notebook. Escolha **uma** das opções abaixo.**Opção A — upload manual** (mais simples, roda uma vez por sessão):

In [ ]:
# OPÇÃO A: upload manual no Colab.# Descomente as duas linhas e selecione o banner.csv do seu computador.# from google.colab import files# files.upload()CSV = Path("banner.csv")

**Opção B — montar o Google Drive** (melhor se você for reabrir o notebook várias vezes):

In [ ]:
# OPÇÃO B: ler direto do Drive.# Ajuste o caminho para onde você guardou o arquivo.# from google.colab import drive# drive.mount("/content/drive")# CSV = Path("/content/drive/MyDrive/manutencao_preditiva/banner.csv")

**Opção C — rodando localmente**, fora do Colab: o CSV já está na raiz do projeto, ao ladode `prep.py`. A célula abaixo aceita qualquer uma das três opções e confirma que o arquivo existe.

In [ ]:
if not CSV.exists():    raise FileNotFoundError(        f"Não encontrei {CSV}. Use a Opção A (upload), a Opção B (Drive) "        "ou ajuste a variável CSV para o caminho correto."    )print(f"Arquivo encontrado: {CSV}  ({CSV.stat().st_size / 1e6:.1f} MB)")

---## 2. Etapa 1 — Carregamento e ordenação temporalO CSV traz 26 colunas. Duas coisas acontecem aqui:1. `created_at` é convertida para `datetime` (sem isso, o campo seria tratado como texto e a   ordenação sairia errada — `"10:00"` viria antes de `"9:00"`).2. As linhas são **ordenadas por tempo**. Isso não é cosmético: todas as etapas seguintes   (segmentação, janelas, cálculo de inclinação) pressupõem que a linha seguinte é o instante   seguinte.

In [ ]:
bruto = pd.read_csv(CSV, parse_dates=["created_at"])print("Formato bruto:", bruto.shape)print("\nColunas:")print(list(bruto.columns))bruto.head(3)

In [ ]:
bruto = bruto.sort_values("created_at").reset_index(drop=True)print("Período coberto:", bruto["created_at"].min(), "→", bruto["created_at"].max())print("Ordenado por tempo:", bruto["created_at"].is_monotonic_increasing)

---## 3. Etapa 2 — SegmentaçãoUm **segmento** é um trecho contínuo de leituras sob a mesma condição de operação — por exemplo,um ensaio inteiro com o rolamento defeituoso instalado.Um novo segmento é aberto quando ocorre **um** destes eventos:- **o rótulo bruto (`fault`) muda** — trocou-se a condição sendo ensaiada;- **o intervalo entre duas leituras passa de 1 hora** — o equipamento ficou parado, então as duas  leituras não pertencem ao mesmo ensaio, mesmo com o mesmo rótulo.O truque de implementação: `new_segment` é um vetor booleano marcando onde começa cada segmento;`cumsum()` sobre ele produz um identificador que só incrementa nessas marcas.

In [ ]:
dt = bruto["created_at"].diff().dt.total_seconds()print(f"Intervalo entre leituras — mediana: {dt.median():.2f} s")print(f"Intervalo entre leituras — média:   {dt.mean():.2f} s  (puxada por longas paradas)")print(f"Intervalos maiores que 1 hora: {int((dt > 3600).sum())}")dt.describe()

> **Anote esse número: a mediana é 2,0 segundos, não 1.** Ele volta a importar na Seção 6, quando> for preciso traduzir "tamanho de janela" em "tempo real coberto".

In [ ]:
raw_label_changed = bruto["fault"].ne(bruto["fault"].shift())new_segment = raw_label_changed | dt.gt(3600)bruto["segment_id"] = new_segment.cumsum()print("Segmentos identificados:", bruto["segment_id"].nunique())

---## 4. Etapa 3 — Normalização de rótulosO campo `fault` foi preenchido manualmente durante os ensaios e chegou inconsistente. Há trêsproblemas distintos:1. **Erros de digitação** — `desabalanceado`, `desbanlanceado`, `ddesbalanceado` e   `dedesbalanceado` são todos `desbalanceado`. Sem unificação, o modelo os trata como classes   diferentes e fragmenta os dados de treino.2. **Rótulos descartáveis** — `teste`, `acelerando`, `new_tes` não descrevem uma condição de   falha e são removidos da análise.3. **Sinônimos** — `baseline` e `normal` são a mesma coisa: motor saudável.Além disso, os rótulos costumam vir com sufixos (`rolamento_inner_2`, `cocked_ensaio3`). A funçãoextrai o **tipo base** procurando o prefixo conhecido.

In [ ]:
def classe_base(rótulo: str) -> str:    """Reduz um rótulo bruto ao seu tipo de falha canônico."""    if pd.isna(rótulo):        return pd.NA    label = str(rótulo).strip()    # 1. Rótulos que não representam falha útil    discarded = {"teste", "acelerando", "new_tes", "new_teste"}    if label in discarded:        return pd.NA    # 2. Erros de digitação recorrentes    replacements = {        "desabalanceado": "desbalanceado",        "desbanlanceado": "desbalanceado",        "ddesbalanceado": "desbalanceado",        "dedesbalanceado": "desbalanceado",        "desabanceado": "desbalanceado",        "desbalanceamento": "desbalanceado",        "normla": "normal",        "mortor_desligado": "motor_desligado",        "cockecocked": "cocked",        "rolamento_comb": "rolamento_combination",    }    for source, target in replacements.items():        if source in label:            label = label.replace(source, target)    label = label.removeprefix("new_")    # 3. Sinônimo: baseline == motor saudável    if label.startswith("baseline"):        return "normal"    # 4. Extração do tipo base pelo prefixo    tipos = [        "motor_desligado", "falta_fase", "rolamento_inner", "rolamento_outer",        "rolamento_ball", "rolamento_combination", "eccentric", "cocked",        "desbalanceado", "desalinhado", "normal", "baseline",        "polia", "correia", "ventoinha",    ]    for tipo in tipos:        if label.startswith(tipo):            return tipo    return label

In [ ]:
# Efeito da normalizaçãoantes = bruto["fault"].nunique()bruto["classe"] = bruto["fault"].apply(classe_base)depois = bruto["classe"].nunique()descartadas = int(bruto["classe"].isna().sum())print(f"Rótulos brutos distintos:   {antes}")print(f"Classes após normalização:  {depois}")print(f"Linhas descartadas:         {descartadas}")pd.DataFrame({"linhas": bruto["classe"].value_counts()})

In [ ]:
dados = bruto.dropna(subset=["classe"]).reset_index(drop=True)print("Formato após descartar linhas sem classe válida:", dados.shape)

---## 5. Etapa 4 — Seleção de sinaisO CSV traz medidas duplicadas em dois sistemas de unidade: `z_rms_velocity_in_s` e`z_rms_velocity_mm_s` são a mesma grandeza, assim como `temperature_f` e `temperature_c`.Manter as duas versões daria ao modelo colunas perfeitamente correlacionadas, sem informaçãonova. Ficamos com o **sistema métrico** e descartamos as colunas em polegada e Fahrenheit.Sobram **18 sinais**.

In [ ]:
SINAIS = [    "z_rms_velocity_mm_s", "temperature_c", "x_rms_velocity_mm_s",    "z_peak_acceleration_g", "x_peak_acceleration_g",    "z_peak_vel_comp_freq_hz", "x_peak_vel_comp_freq_hz",    "z_rms_acceleration_g", "x_rms_acceleration_g",    "z_kurtosis", "x_kurtosis",    "z_crest_factor", "x_crest_factor",    "z_peak_velocity_mm_s", "x_peak_velocity_mm_s",    "z_high_freq_rms_accel_g", "x_high_freq_rms_accel_g",    "rpm",]print(f"{len(SINAIS)} sinais mantidos")pd.DataFrame({"sinal": SINAIS})

### 5.1 A função completa de preparaçãoAs Seções 2 a 5 correspondem, juntas, à função `preparar_dataframe` do projeto(arquivo `prep.py`). Ela é reproduzida abaixo na íntegra para que o restante do notebookuse exatamente o mesmo código do sistema.

In [ ]:
def preparar_dataframe(df: pd.DataFrame) -> pd.DataFrame:    """Ordena, segmenta, normaliza rótulos e seleciona colunas."""    df = df.copy()    # Etapa 1 — ordenação temporal    if "created_at" in df.columns:        df["created_at"] = pd.to_datetime(df["created_at"])        df = df.sort_values("created_at").reset_index(drop=True)    else:        df = df.reset_index(drop=True)    # Etapa 2 — segmentação    if "fault" in df.columns:        dt = df["created_at"].diff().dt.total_seconds()        raw_label_changed = df["fault"].ne(df["fault"].shift())        new_segment = raw_label_changed | dt.gt(3600)        df["segment_id"] = new_segment.cumsum()    else:        df["segment_id"] = 0    # Etapa 3 — normalização de rótulos    if "fault" in df.columns:        df["classe"] = df["fault"].apply(classe_base)        df = df.dropna(subset=["classe"]).reset_index(drop=True)    else:        df["classe"] = pd.NA    # Etapa 4 — seleção de sinais    feature_cols = []    if "created_at" in df.columns:        feature_cols.append("created_at")    feature_cols.extend([*SINAIS, "classe", "segment_id"])    df = df.loc[:, feature_cols]    return dfdef carregar(caminho=None) -> pd.DataFrame:    df = pd.read_csv(caminho or CSV, parse_dates=["created_at"])    return preparar_dataframe(df)

In [ ]:
df = carregar()print("Formato final:", df.shape)print("Valores ausentes:", int(df.isna().sum().sum()))print("Classes:", df["classe"].nunique())print("Segmentos:", df["segment_id"].nunique())df.head()

---## 6. Etapa 5 — Fatiar em janelasTemos um dataframe corrido de leituras, com os segmentos já identificados. Antes de resumirqualquer coisa, é preciso decidir **o que será resumido** — ou seja, recortar os pedaços.> **Ordem importa.** O recorte vem **antes** do resumo. No código, `criar_amostras` primeiro> fatia (`grupo.iloc[inicio : inicio + tamanho]`) e só então chama `features_janela` sobre a> fatia. A Etapa 6 trata do resumo.Há duas estratégias de recorte:**Modo segmento** — cada segmento inteiro é um pedaço.Preserva o contexto completo do ensaio, mas produz poucas amostras (uma por segmento) e trataigualmente segmentos de 50 e de 6.000 leituras.**Modo janela** — cada segmento é fatiado em blocos de tamanho fixo, com 50% de sobreposição.Produz muito mais amostras e uniformiza a duração. Uma janela **nunca cruza a fronteira de umsegmento**, para não misturar duas condições de operação no mesmo pedaço.### 6.1 Tamanho da janela e tempo realComo a mediana do intervalo entre leituras é **2,0 s** (Seção 3), o tamanho da janela em amostrasnão coincide com o tempo coberto:| Janela (amostras) | Tempo real aproximado ||---|---|| 30 | ~60 s || 90 | ~180 s (3 min) || **180** | **~360 s (6 min)** || 360 | ~720 s (12 min) |A configuração adotada foi **180 amostras**, com passo de 90.

In [ ]:
# Tamanho da janela, em número de amostras consecutivas.# O passo é metade da janela, o que gera 50% de sobreposição.JANELA_TAMANHO = 180JANELA_PASSO = JANELA_TAMANHO // 2print(f"Janela: {JANELA_TAMANHO} amostras  |  passo: {JANELA_PASSO}  |  sobreposição: 50%")print(f"Tempo real coberto: ~{JANELA_TAMANHO * 2.0 / 60:.0f} minutos por janela")

### 6.2 O recorte, isoladoA função abaixo faz **apenas** o fatiamento, sem resumir nada. Ela existe para deixar visívelque, nesta etapa, cada janela ainda é um bloco de leituras — não uma lista de números.

In [ ]:
def recortar_janelas(df, tamanho=None, passo=None):    """Devolve a lista de janelas cruas, sem calcular features."""    tamanho = JANELA_TAMANHO if tamanho is None else tamanho    passo = (tamanho // 2) if passo is None else passo    janelas = []    for segment_id, grupo in df.groupby("segment_id", sort=True):        # Segmentos curtos demais não geram nenhuma janela        if len(grupo) < tamanho:            continue        for inicio in range(0, len(grupo) - tamanho + 1, passo):            janelas.append(grupo.iloc[inicio : inicio + tamanho])    return janelasjanelas_cruas = recortar_janelas(df)print(f"Janelas recortadas: {len(janelas_cruas)}")print(f"Formato de cada uma: {janelas_cruas[0][SINAIS].shape} (linhas × colunas)")

In [ ]:
# Uma janela por dentro: ainda é um bloco de leiturasprimeira = janelas_cruas[0]print(f"Ensaio {int(primeira['segment_id'].iloc[0])} | classe: {primeira['classe'].iloc[0]}")primeira[SINAIS].head(10)

**Nada foi resumido ainda.** Cada janela continua sendo um bloco de 180 × 18. Transformar cadabloco em uma linha só é o trabalho da Etapa 6.---## 7. Etapa 6 — Resumir cada janela em númerosUm modelo tabular como o Random Forest não recebe blocos nem séries temporais; ele recebe **umalinha por exemplo**. Cada janela de 180 × 18 precisa virar **uma única linha**.Para cada um dos 18 sinais calculamos **5 estatísticas**:| Estatística | O que captura | Por que importa ||---|---|---|| **mediana** | nível típico do sinal | robusta a picos isolados, diferente da média || **desvio padrão** | quanto o sinal oscila | falhas de rolamento elevam a variabilidade || **inclinação** | tendência de subida/descida | separa degradação progressiva de regime estável || **amplitude** (máx − mín) | excursão total | sensível a eventos extremos || **IQR** (p90 − p10) | dispersão robusta | como a amplitude, mas ignorando os 20% extremos |18 sinais × 5 estatísticas = **90 colunas**.> A inclinação é calculada por regressão linear em forma fechada. Centrando o tempo em zero> (`t = arange(n) - (n-1)/2`), o intercepto some da conta e a inclinação vira `Σ(t·x) / Σ(t²)`,> que é mais barato do que ajustar um modelo por trecho.

In [ ]:
def _features_resumo(dados: pd.DataFrame) -> np.ndarray:    """Resume um bloco de sinais em 5 estatísticas por coluna."""    valores = dados[SINAIS].to_numpy(dtype=float)    n = len(dados)    # Tempo centrado em zero: elimina o intercepto do cálculo da inclinação    if n <= 1:        t = np.array([0.0] * n, dtype=float)        denom = 1.0    else:        t = np.arange(n) - (n - 1) / 2        denom = float(np.dot(t, t))    features = []    for coluna in range(valores.shape[1]):        x = valores[:, coluna]        slope = float(np.dot(t, x) / denom) if denom > 0 else 0.0        features.extend([            float(np.median(x)),                                    # mediana            float(np.std(x)),                                       # desvio padrão            slope,                                                  # inclinação            float(np.max(x) - np.min(x)),                           # amplitude            float(np.percentile(x, 90) - np.percentile(x, 10)),     # IQR        ])    return np.array(features, dtype=float)ESTATISTICAS = ["median", "std", "slope", "range", "iqr"]NOMES_FEATURES = [f"{stat}_{sinal}" for sinal in SINAIS for stat in ESTATISTICAS]print(f"{len(SINAIS)} sinais × {len(ESTATISTICAS)} estatísticas = {len(NOMES_FEATURES)} colunas")

### 7.1 A transformação, em formato de tabelaRepare no formato de entrada e de saída. **A janela entra como um bloco de 180 linhas e saicomo uma linha só, com 90 colunas.** Os nomes das features são **nomes de coluna**, não linhasde uma lista.

In [ ]:
# ENTRA: um bloco de 180 linhas × 18 colunasprint("ENTRA:", primeira[SINAIS].shape, "(linhas × colunas)")primeira[SINAIS].head(5)

In [ ]:
# SAI: uma linha só, com 90 colunasvetor = _features_resumo(primeira)uma_linha = pd.DataFrame([vetor], columns=NOMES_FEATURES, index=["janela nº 1"])print("SAI:", uma_linha.shape, "(linhas × colunas)")uma_linha

A tabela acima tem 90 colunas e não cabe na tela. Para conferir os valores com calma, a célulaabaixo mostra **os mesmos 90 números virados de lado** — é a mesma linha, girada na vertical.Na tabela real, cada um desses nomes é uma coluna.

In [ ]:
# A MESMA linha acima, apenas girada para facilitar a leiturapd.DataFrame({    "nome da coluna": NOMES_FEATURES,    "valor nesta janela": vetor,    "sinal de origem": [s for s in SINAIS for _ in ESTATISTICAS],    "estatística": ESTATISTICAS * len(SINAIS),}).head(15)

> **Cuidado com a coincidência numérica:** a saída tem 90 colunas porque 18 × 5 = 90.> Esse 90 **não tem relação** com o tamanho da janela (que também chegou a ser 90 durante os> testes). Mudar o tamanho da janela não muda o número de colunas.### 7.2 Juntando recorte e resumoA função `criar_amostras` faz as duas etapas de uma vez: fatia (Etapa 5) e resume (Etapa 6).É ela que o projeto usa de fato.

In [ ]:
def features_janela(janela: pd.DataFrame, tamanho=None) -> np.ndarray:    tamanho = JANELA_TAMANHO if tamanho is None else tamanho    if len(janela) != tamanho:        raise ValueError(f"A janela deve conter exatamente {tamanho} amostras")    return _features_resumo(janela)def features_segmento(segmento: pd.DataFrame) -> np.ndarray:    if segmento.empty:        raise ValueError("O segmento deve conter pelo menos uma amostra")    return _features_resumo(segmento)def criar_amostras(df, modo="segmento", tamanho=None, passo=None):    """Fatia (Etapa 5) e resume (Etapa 6) em uma passada só."""    tamanho = JANELA_TAMANHO if tamanho is None else tamanho    passo = (tamanho // 2) if passo is None else passo    modo = str(modo).lower()    amostras = []    for segment_id, grupo in df.groupby("segment_id", sort=True):        if grupo.empty:            continue        # Modo segmento: o segmento inteiro é o pedaço        if modo in {"segmento", "sem_janela", "segmento_completo"}:            amostras.append({                "features": features_segmento(grupo),                "classe": grupo["classe"].iloc[0],                "segment_id": int(segment_id),            })            continue        # Modo janela: segmentos curtos demais são descartados por inteiro        if len(grupo) < tamanho:            continue        for inicio in range(0, len(grupo) - tamanho + 1, passo):            janela = grupo.iloc[inicio : inicio + tamanho]     # 1º recorta            amostras.append({                "features": features_janela(janela, tamanho=tamanho),   # 2º resume                "classe": janela["classe"].iloc[0],                "segment_id": int(segment_id),            })    return pd.DataFrame(amostras)

In [ ]:
amostras_segmento = criar_amostras(df, modo="segmento")amostras_janela = criar_amostras(df, modo="janela")print(f"Modo segmento: {len(amostras_segmento):>6} amostras | "      f"{amostras_segmento['classe'].nunique()} classes")print(f"Modo janela:   {len(amostras_janela):>6} amostras | "      f"{amostras_janela['classe'].nunique()} classes")

In [ ]:
# A tabela final que vai para o modelo: uma linha por janela, 90 colunas de númerostabela_final = pd.DataFrame(    np.vstack(amostras_janela["features"].to_list()), columns=NOMES_FEATURES)tabela_final.insert(0, "classe", amostras_janela["classe"].to_numpy())tabela_final.insert(1, "segment_id", amostras_janela["segment_id"].to_numpy())print("Formato:", tabela_final.shape, "(linhas × colunas)")tabela_final.head()

> Repare que o modo janela com 180 amostras produz **13 classes, não 14**. A classe `falta_fase`> desaparece: nenhum segmento dela tem 180 leituras consecutivas. A Seção 11 detalha esse custo.---## 8. Etapa 7 — Modelo**Random Forest Classifier**, com `n_estimators=400`, `random_state=0`, `n_jobs=-1`.A escolha se justifica pelo formato do problema:- lida bem com **features em escalas diferentes** (Hz, °C, g, mm/s) sem exigir normalização;- é robusto a **classes desbalanceadas**, que é o caso aqui;- captura **interações não lineares** entre sinais sem que seja preciso especificá-las;- fornece **importância de features**, útil para interpretar o que sustenta a decisão.`random_state=0` fixa a semente para que os resultados sejam reproduzíveis.

In [ ]:
X = np.vstack(amostras_janela["features"].to_list())y = amostras_janela["classe"].to_numpy()groups = amostras_janela["segment_id"].to_numpy()print("Matriz de entrada X:", X.shape, " (amostras × features)")print("Vetor alvo y:       ", y.shape)print("Grupos (segmentos): ", len(np.unique(groups)), "segmentos distintos")

---## 9. Etapa 8 — Validação cruzada (e por que existem duas)Este é o ponto conceitualmente mais importante do projeto.### 9.1 Validação aleatória — `StratifiedKFold`Embaralha todas as amostras e divide em 5 folds, preservando a proporção das classes.**O problema:** as janelas vêm de segmentos, e janelas do mesmo segmento são quase idênticas —mesmo motor, mesmo ensaio, minutos de diferença, e ainda com 50% de sobreposição entre janelasvizinhas. Ao embaralhar, janelas do mesmo segmento caem em treino **e** em teste. O modeloreconhece o segmento que já viu, e não a falha. Isso é **vazamento de dados**, e o resultado saiinflado.### 9.2 Validação por segmento — `StratifiedGroupKFold`Garante que **todas as janelas de um mesmo segmento fiquem do mesmo lado** da divisão. O modelo étestado em segmentos que nunca viu.Essa é a pergunta que importa na prática: *o sistema funciona num motor novo, num ensaio novo?*> **A acurácia por segmento é a métrica honesta.** A aleatória é mantida no projeto como> referência — a distância entre as duas mede o tamanho do vazamento.

In [ ]:
def avaliar(X, y, groups, n_estimators=400, n_splits=5):    """Roda as duas validações cruzadas e devolve os resultados."""    validadores = [        (StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=0), "aleatoria"),        (StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=0), "por_segmento"),    ]    resultados = []    for splitter, nome in validadores:        scores = []        for train_idx, test_idx in splitter.split(X, y, groups):            modelo = RandomForestClassifier(                n_estimators=n_estimators, random_state=0, n_jobs=-1            )            modelo.fit(X[train_idx], y[train_idx])            preds = modelo.predict(X[test_idx])            scores.append(accuracy_score(y[test_idx], preds))        resultados.append({            "nome": nome,            "folds": [float(s) for s in scores],            "media": float(np.mean(scores)),            "desvio": float(np.std(scores)),        })    return resultados

In [ ]:
# ATENÇÃO: esta célula treina 10 florestas de 400 árvores. Leva alguns minutos.resultados = avaliar(X, y, groups)for r in resultados:    folds = "  ".join(f"{s:.3f}" for s in r["folds"])    print(f"{r['nome']:>13}: média {r['media']:.4f}  (desvio {r['desvio']:.3f})   folds: {folds}")

**Resultado obtido com a configuração adotada (janela = 180):**| Validação | Acurácia média ||---|---|| Aleatória (`StratifiedKFold`) | **0,921** || Por segmento (`StratifiedGroupKFold`) | **0,438** |A distância entre as duas — mais de 48 pontos — é o vazamento medido. O número a reportar comodesempenho real do sistema é **0,438**.

### 9.3 Onde o modelo erraA acurácia sozinha não diz quais falhas são confundidas entre si. O relatório por classe e amatriz de confusão abaixo usam um único fold da validação por segmento.

In [ ]:
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=0)train_idx, test_idx = next(sgkf.split(X, y, groups))modelo = RandomForestClassifier(n_estimators=400, random_state=0, n_jobs=-1)modelo.fit(X[train_idx], y[train_idx])preds = modelo.predict(X[test_idx])print(f"Acurácia neste fold: {accuracy_score(y[test_idx], preds):.4f}\n")print(classification_report(y[test_idx], preds, zero_division=0))

In [ ]:
rotulos = sorted(np.unique(np.concatenate([y[test_idx], preds])))matriz = confusion_matrix(y[test_idx], preds, labels=rotulos)pd.DataFrame(matriz, index=[f"real: {r}" for r in rotulos],             columns=[f"prev: {r}" for r in rotulos])

In [ ]:
# Quais sinais o modelo mais usaimportancias = (    pd.DataFrame({"feature": NOMES_FEATURES, "importancia": modelo.feature_importances_})    .sort_values("importancia", ascending=False)    .head(15)    .reset_index(drop=True))importancias

---## 10. Experimento — varredura do tamanho da janelaO tamanho da janela foi testado em quatro valores: **30, 90, 180 e 360**. Cada um foi avaliadocom as duas validações.### 10.1 Resultados medidos| Janela | Nº de janelas | Classes | Segmentos usados | Aleatória | Por segmento ||---|---|---|---|---|---|| 30 | 10.825 | 14 | 229 / 234 | 0,922 | 0,433 || 90 | 3.401 | 14 | 172 / 234 | 0,914 | **0,450** || **180** | **1.559** | **13** | **75 / 234** | **0,921** | **0,438** || 360 | 721 | 12 | 63 / 234 | 0,827 | 0,418 |### 10.2 Leitura dos resultados**A faixa é estreita.** Na validação por segmento, os quatro valores ficam entre 0,418 e 0,450 —3 pontos de amplitude total. O tamanho da janela **não é a alavanca** deste problema.**A partir de 90 a tendência é monotônica de queda:** 0,450 → 0,438 → 0,418.**Com 180, a dispersão entre folds explode.** Os folds vão de 0,346 a 0,523 — quase 18 pontos.Isso é efeito de amostra pequena, não de sinal: com 75 segmentos, cada fold de teste tem ~15grupos, e qual segmento cai onde passa a dominar o resultado. A diferença entre 0,438 e 0,450não é distinguível dado esse ruído.**Com 360, até a métrica otimista cede.** A acurácia aleatória cai de ~0,92 para 0,827 — com 721janelas para 90 features e 12 classes, não há amostras suficientes nem para o resultado inflado.### 10.3 Reproduzir a varreduraA célula abaixo refaz o experimento inteiro. **É cara** — 40 florestas de 400 árvores, dezenas deminutos. Deixe-a comentada a menos que queira reproduzir os números.

In [ ]:
# Descomente para reproduzir a varredura completa.# linhas = []# for tam in (30, 90, 180, 360):#     a = criar_amostras(df, modo="janela", tamanho=tam, passo=tam // 2)#     Xi = np.vstack(a["features"].to_list())#     yi = a["classe"].to_numpy()#     gi = a["segment_id"].to_numpy()#     res = avaliar(Xi, yi, gi)#     linhas.append({#         "janela": tam,#         "janelas": len(a),#         "classes": a["classe"].nunique(),#         "segmentos": a["segment_id"].nunique(),#         "aleatoria": round(res[0]["media"], 4),#         "por_segmento": round(res[1]["media"], 4),#     })#     print(linhas[-1])## pd.DataFrame(linhas)

---## 11. Custo de descarte por tamanho de janelaHá um custo que a acurácia não mostra. No modo janela, **um segmento com menos leituras que otamanho da janela é descartado por inteiro** — não gera nenhuma amostra. Quanto maior a janela,mais segmentos se perdem.A célula abaixo é barata (não treina nada) e quantifica essa perda.

In [ ]:
tam_seg = df.groupby("segment_id").size()total_seg = len(tam_seg)total_linhas = len(df)todas_classes = set(df["classe"].unique())linhas = []for t in (30, 90, 180, 360):    ok = tam_seg[tam_seg >= t]    ids = ok.index    classes_ok = set(df.loc[df["segment_id"].isin(ids), "classe"].unique())    perdidas = sorted(todas_classes - classes_ok)    linhas.append({        "janela": t,        "segmentos_usados": len(ok),        "descartados": total_seg - len(ok),        "%_descartado": round(100 * (total_seg - len(ok)) / total_seg, 1),        "%_linhas_cobertas": round(100 * int(ok.sum()) / total_linhas, 1),        "classes": len(classes_ok),        "classes_perdidas": ", ".join(perdidas) if perdidas else "—",    })print(f"Segmentos: {total_seg} | tamanho — mín {tam_seg.min()}, "      f"mediana {int(tam_seg.median())}, máx {tam_seg.max()}\n")pd.DataFrame(linhas)

### 11.1 Duas leituras que divergem**Em linhas de dado**, a perda parece pequena: mesmo com janela de 180, 89,7% das leiturascontinuam cobertas, porque os segmentos descartados são todos curtos.**Em segmentos, a perda é grave** — e é a que importa. A validação `StratifiedGroupKFold` agrupapor segmento: 172 segmentos (janela 90) dão folds de ~34 grupos, enquanto 75 segmentos(janela 180) dão folds de ~15. É exatamente isso que explica a dispersão entre folds observadana Seção 10.### 11.2 O degrau entre 90 e 180A mediana do tamanho dos segmentos é **150 leituras**. A janela de 180 cai logo acima dessamediana, então mais da metade dos segmentos morre de uma vez: o descarte salta de **26,5%**(janela 90) para **67,9%** (janela 180). É o degrau mais caro de toda a faixa testada.### 11.3 Classes perdidas- **Janela 180:** `falta_fase` desaparece — o modelo passa a 13 classes e não consegue mais  detectar falta de fase. `motor_desligado` fica com 2 janelas.- **Janela 360:** perdem-se `falta_fase` **e** `motor_desligado` — 12 classes.> Se falta de fase for uma condição que o sistema precisa detectar em produção, a janela de 180> não atende a esse requisito.

---## 12. Conclusões, decisão tomada e pendências### 12.1 Configuração adotada| Parâmetro | Valor ||---|---|| Tamanho da janela | **180 amostras** (~6 min) || Passo | 90 (sobreposição de 50%) || Features | 90 (18 sinais × 5 estatísticas) || Modelo | RandomForestClassifier(n_estimators=400, random_state=0) || **Acurácia (por segmento)** | **0,438** || Acurácia (aleatória, inflada) | 0,921 |### 12.2 O que os experimentos mostraram**O tamanho da janela não é o gargalo.** Quatro valores entre 30 e 360 variam a acurácia porsegmento de 0,418 a 0,450 — 3 pontos. Continuar varrendo tamanhos não deve render mais.**O gargalo é a generalização entre segmentos.** Em todas as configurações a acurácia aleatóriafica 45–50 pontos acima da acurácia por segmento. Esse abismo é a informação central destetrabalho: o modelo acerta quando vê janelas do mesmo segmento no treino e erra quando precisageneralizar para um segmento novo. É assinatura de segmento sendo memorizada — condiçõesespecíficas daquele ensaio (montagem, temperatura ambiente, rotação exata) em vez do padrão dafalha. Nenhum tamanho de janela corrige isso.### 12.3 Caminhos com maior retorno esperado1. **Features invariantes ao segmento** — trabalhar com razões e valores relativos ao regime do   próprio segmento, em vez de níveis absolutos que carregam a assinatura do ensaio.2. **Normalização por segmento** — centrar cada segmento na própria linha de base antes de   extrair estatísticas.3. **Revisar a definição de segmento** — verificar se o critério atual (rótulo + 1 hora) não está   agrupando ensaios que deveriam ser distintos, ou separando o que é contínuo.4. **Tratar as classes raras** — `falta_fase` e `motor_desligado` têm poucos segmentos e são as   primeiras a se perder quando a janela cresce.### 12.4 Pendências conhecidas- **Dois testes defasados** em `tests/test_prep.py`: `test_carregar_dataframe` espera 166.796  linhas e 27 colunas (o real é 166.688 × 21) e `test_segmentacao` espera 240 segmentos (o real é  234). São expectativas antigas, anteriores às etapas de normalização de rótulos e seleção de  sinais. Falham hoje e ainda não foram corrigidas.- **`falta_fase` não é detectável** na configuração de 180 amostras (Seção 11.3).### 12.5 Arquivos do projeto| Arquivo | Responsabilidade ||---|---|| `prep.py` | Carga, segmentação, normalização, features, janelamento || `avaliacao.py` | Validação cruzada com as duas estratégias || `sistema.py` | Consulta ao modelo treinado (opera em modo segmento) || `app.py` | Interface Streamlit de visualização do pipeline || `interface_app.py` | Interface de console || `tests/` | Testes unitários |